<a href="https://colab.research.google.com/github/kotatech24/AIML/blob/main/Another_copy_of_Building_Multi_AI_Agents_Chatbots_With_LangGraph_Groq_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Agent Travel & Hotel Advisor — Groq + LangGraph v1

This version is converted from the original Anthropic notebook to the current LangChain/LangGraph agent API and Groq. It preserves the Travel Advisor → Human → Hotel Advisor workflow while avoiding the deprecated `create_react_agent` API.

**Model:** `openai/gpt-oss-120b` on Groq. Groq currently lists this model as supporting tool use and agentic workflows.

In [1]:
!pip install -U langchain langgraph langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 28.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.11
    Uninstalling langgraph-1.2.11:
      Successfully uninstalled langgraph-1.2.11
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.18
    Uninstalling langchain-1.3.18:
      Successfully uninstalled langchain-1.3.18


## 1. Load the Groq API key

This works in a local Jupyter/Miniconda notebook and does not depend on Google Colab `userdata`.

In [2]:
import os
from getpass import getpass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("GROQ_API_KEY loaded:", bool(os.getenv("GROQ_API_KEY")))

Enter your Groq API key: ··········
GROQ_API_KEY loaded: True


In [3]:
from typing import Literal

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool

from langgraph.graph import MessagesState, StateGraph, START
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import MemorySaver

from langchain_core.messages import HumanMessage, AIMessage

print("Imports successful")

Imports successful


In [4]:
# Groq model
# GPT-OSS 120B is currently recommended by Groq for agentic/tool-use workloads.
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_retries=2,
)

print("Model initialized:", model.model_name)

Model initialized: openai/gpt-oss-120b


In [5]:
# Quick model test — this does not use any tools.
response = model.invoke("Say hello in one short sentence.")
print(response.content)

Hello!


In [6]:
class MultiAgentState(MessagesState):
    last_active_agent: Literal["travel_advisor", "hotel_advisor"]


In [7]:
@tool
def get_travel_recommendations(query: str) -> str:
    """Return travel destination recommendations for the user's travel request."""
    q = query.lower()

    if "caribbean" in q:
        return (
            "For a warm Caribbean vacation, consider Jamaica, the Bahamas, "
            "Barbados, Grenada, or Saint Lucia."
        )
    if "beach" in q:
        return "For a beach vacation, I recommend Hawaii, Bali, or the Maldives."
    if "mountain" in q:
        return "For a mountain getaway, consider the Swiss Alps, the Rockies, or the Himalayas."
    if "city" in q:
        return "For a city break, consider New York, Paris, or Tokyo."
    return "I recommend destinations such as Hawaii, Bali, Paris, Jamaica, or the Bahamas."


@tool
def get_hotel_recommendations(query: str) -> str:
    """Return hotel recommendations based on the user's destination or hotel request."""
    q = query.lower()

    if "caribbean" in q or "jamaica" in q:
        return (
            "For Jamaica, consider Half Moon in Montego Bay or Jamaica Inn in Ocho Rios. "
            "Montego Bay is a good choice for beaches and resorts."
        )
    if "beach" in q:
        return "For a beach stay, consider The Ritz-Carlton, Bali or Four Seasons Maui."
    if "city" in q:
        return "For a city stay, consider The Peninsula Hong Kong or The Savoy London."
    return "Consider well-rated properties such as The Ritz-Carlton or Four Seasons, depending on the destination."

In [8]:
travel_advisor = create_agent(
    model=model,
    tools=[get_travel_recommendations],
    system_prompt=(
        "You are the Travel Advisor. "
        "Help users choose travel destinations. "
        "Use get_travel_recommendations when destination recommendations are needed. "
        "Do not provide hotel recommendations yourself; the outer LangGraph will route hotel requests to the Hotel Advisor. "
        "Keep answers concise and useful."
    ),
    name="travel_advisor",
)

hotel_advisor = create_agent(
    model=model,
    tools=[get_hotel_recommendations],
    system_prompt=(
        "You are the Hotel Advisor. "
        "Recommend hotels and explain the area where appropriate. "
        "Use get_hotel_recommendations when hotel recommendations are needed. "
        "You may also answer simple questions about activities near the recommended hotel using the conversation context. "
        "Keep answers concise and useful."
    ),
    name="hotel_advisor",
)

print("Travel and Hotel agents created")

Travel and Hotel agents created


In [9]:
def call_travel_advisor(state: MultiAgentState) -> Command:
    # create_agent expects a messages state. We intentionally pass only messages.
    response = travel_advisor.invoke({"messages": state["messages"]})

    return Command(
        update={
            "messages": response["messages"],
            "last_active_agent": "travel_advisor",
        },
        goto="human",
    )


def call_hotel_advisor(state: MultiAgentState) -> Command:
    response = hotel_advisor.invoke({"messages": state["messages"]})

    return Command(
        update={
            "messages": response["messages"],
            "last_active_agent": "hotel_advisor",
        },
        goto="human",
    )

In [10]:
def route_after_human(user_input: str, active_agent: str) -> str:
    """Simple deterministic router for this teaching example."""
    text = user_input.lower()

    hotel_words = [
        "hotel", "resort", "stay", "accommodation", "room",
        "where should i stay", "place to stay",
    ]
    travel_words = [
        "destination", "where should i go", "where can i go",
        "travel", "trip", "vacation", "holiday",
    ]

    if any(word in text for word in hotel_words):
        return "hotel_advisor"

    if any(word in text for word in travel_words):
        return "travel_advisor"

    # Otherwise continue with whichever advisor was active.
    return active_agent


def human_node(state: MultiAgentState) -> Command:
    user_input = interrupt(value="Ready for user input.")
    active_agent = state.get("last_active_agent", "travel_advisor")
    next_agent = route_after_human(user_input, active_agent)

    return Command(
        update={
            "messages": [HumanMessage(content=user_input)],
        },
        goto=next_agent,
    )

In [11]:
builder = StateGraph(MultiAgentState)

builder.add_node("travel_advisor", call_travel_advisor)
builder.add_node("hotel_advisor", call_hotel_advisor)
builder.add_node("human", human_node)

builder.add_edge(START, "travel_advisor")

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print("Graph compiled successfully")

Graph compiled successfully


In [12]:
import uuid

thread_config = {
    "configurable": {
        "thread_id": str(uuid.uuid4())
    }
}

inputs = [
    {
        "messages": [
            {
                "role": "user",
                "content": "I wanna go somewhere warm in the Caribbean"
            }
        ]
    },
    Command(
        #resume="Could you recommend a nice hotel in one of the areas and tell me which area it is?"
        resume=input("Your query: ")
    ),
    Command(
        resume="I like the first one. Could you recommend something to do near the hotel?"
        #resume=input("Your query: ")
    ),
]

for idx, user_input in enumerate(inputs):
    print(f"\n--- Conversation Turn {idx + 1} ---\n")

    if isinstance(user_input, dict):
        print("User:", user_input["messages"][0]["content"])
    else:
        print("User:", user_input.resume)

    for update in graph.stream(
        user_input,
        config=thread_config,
        stream_mode="updates",
    ):
        for node_id, value in update.items():
            if not isinstance(value, dict):
                continue

            messages = value.get("messages", [])
            if not messages:
                continue

            last_message = messages[-1]

            if isinstance(last_message, AIMessage):
                print(f"{node_id}: {last_message.content}")

Your query: Please share hotel information

--- Conversation Turn 1 ---

User: I wanna go somewhere warm in the Caribbean
travel_advisor: Here are a few warm Caribbean spots, each with its own vibe:

| Destination | Highlights | Best time to visit |
|------------|------------|--------------------|
| **Jamaica** | Reggae culture, Dunn’s River Falls, coffee tours, vibrant nightlife in Montego Bay & Negril | Dec‑Apr (dry season) |
| **Bahamas** | Crystal‑clear waters, swimming with pigs on Exuma, luxury resorts on Nassau & Paradise Island | Nov‑Apr |
| **Barbados** | Sandy beaches, historic Bridgetown, surf at Bathsheba, lively festivals | Dec‑Apr |
| **Grenada** | “Spice Island” cocoa & nutmeg farms, underwater sculpture park, quieter beaches | Jan‑May |
| **Saint Lucia** | Dramatic Pitons, rainforest hikes, luxury spa resorts, sailing | Dec‑Apr |

All of these stay comfortably warm (mid‑70s °F / low‑20s °C) year‑round. Let me know if any of them catch your eye or if you have specific ac

## What changed from the LMS notebook?

1. **Anthropic was removed** — no `langchain-anthropic`, `anthropic`, or `ChatAnthropic`.
2. **Groq is used through `ChatGroq`**.
3. **`create_react_agent` was replaced by `langchain.agents.create_agent`**, which is the current LangChain v1 API.
4. The old custom handoff tools were removed. The **outer LangGraph** now performs the advisor-to-advisor routing.
5. Human-in-the-loop is still implemented with `interrupt()` and `Command(resume=...)`.
6. `MemorySaver` is retained so the conversation remains in the same thread.
7. The original three-turn demonstration is preserved: **Caribbean destination → hotel → activity near hotel**.

LangChain documents `create_agent` as the current standard agent API, replacing `langgraph.prebuilt.create_react_agent`. citeturn0search1turn0search5

Groq's LangChain integration supports tool calling, and Groq currently lists GPT-OSS 120B as a tool-use/agentic model. citeturn0search2turn2search0